# Asistente Bancario — Fine-tuning Gemma 3 4B con QLoRA

## GenAI Lifecycle: Modelo + Adaptación

**Selección del modelo:** Gemma 3 4B Instruct
- Multilingüe (soporta español nativamente)
- 4B parámetros → cabe en T4 con quantización 4-bit
- Unsloth lo optimiza (~2x velocidad de entrenamiento)

**Estrategia:** QLoRA (Quantized LoRA)
- Solo entrenamos ~1-2% de los parámetros
- El modelo base queda congelado en 4-bit
- Adapter LoRA ocupa ~50MB vs ~8GB del modelo completo

**Coste:** $0 (Google Colab Free)

In [ ]:

!pip install -qU unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.7/405.7 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.8/110.8 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [7]:

import unsloth
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template, standardize_data_formats, train_on_responses_only
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
from transformers import TextStreamer
from huggingface_hub import notebook_login
import torch

notebook_login()

## 2. Cargar el modelo

Elegí Gemma 3 4B porque es multilingüe (entiende español sin problemas), tiene un buen balance tamaño/calidad, y Unsloth lo soporta bien. Lo cargo en 4-bit para que entre en la T4 de Colab Free — si lo cargara en fp16 no alcanzaría la memoria.

In [2]:
model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-3-4b-it",
    max_seq_length=2048,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
)

==((====))==  Unsloth 2026.1.4: Fast Gemma3 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: Gemma3 does not support SDPA - switching to fast eager.


## 3. Configurar LoRA

Uso r=8 y alpha=8 que es una configuración conservadora. Con r más alto (16, 32) el modelo tiene más capacidad de aprender pero tarda más y usa más memoria. Para un primer intento con Colab Free me pareció mejor ir con valores bajos y ver si alcanza. Dropout en 0 porque el dataset es chico y no quiero que pierda señal.

In [3]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=8,
    lora_alpha=8,
    lora_dropout=0,
    bias="none",
    random_state=3407,
)

Unsloth: Making `model.base_model.model.model.language_model` require gradients


In [4]:
HF_USERNAME = "testlegadoss"
dataset = load_dataset(f"{HF_USERNAME}/retail-banking-chatbot-es", split="train")

# Configurar chat template de Gemma 3
tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")

# Convertir a formato conversacional
def format_to_conversations(example):
    return {
        "conversations": [
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": example["response"]},
        ]
    }

dataset = dataset.map(format_to_conversations)
dataset = standardize_data_formats(dataset)

# Aplicar chat template
def apply_chat_template(examples):
    texts = tokenizer.apply_chat_template(examples["conversations"])
    return {"text": texts}

dataset = dataset.map(apply_chat_template, batched=True)

print(f"Dataset listo: {len(dataset)} samples")
print(f"Ejemplo de texto formateado:\n{dataset[0]['text'][:500]}")

Map:   0%|          | 0/4495 [00:00<?, ? examples/s]

Unsloth: Standardizing formats (num_proc=2):   0%|          | 0/4495 [00:00<?, ? examples/s]

Map:   0%|          | 0/4495 [00:00<?, ? examples/s]

Dataset listo: 4495 samples
Ejemplo de texto formateado:
<bos><start_of_turn>user
Necesito bloquear una tarjeta de crédito, ¿dónde podría hacerlo?<end_of_turn>
<start_of_turn>model
Estoy aquí para ayudarle a bloquear su tarjeta de crédito. Para bloquear su tarjeta de crédito, puede seguir estos pasos: 1. Visite nuestro sitio web o aplicación móvil. 2. Inicie sesión en su cuenta utilizando su nombre de usuario y contraseña. 3. Navegue a la sección "Cuenta" o "Gestión de la tarjeta". 4. Busque la opción de "Bloquear tarjeta" o "Report Lost/Stolen Card".


## 5. Entrenamiento

Pongo 60 steps como punto de partida — se puede subir si hay tiempo en Colab. Lo importante es `train_on_responses_only`: esto hace que el modelo solo aprenda de las respuestas del asistente y no de las preguntas del usuario. Si no hiciera esto, el modelo "gastaría" capacidad aprendiendo a generar preguntas también, que no es lo que quiero.

In [8]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=None,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,  # Ajustar según tiempo disponible en Colab
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",
        output_dir="banking-assistant-gemma3",
    ),
)

# Solo entrenar en respuestas del asistente (no en prompts del usuario)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)

trainer_stats = trainer.train()

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/4495 [00:00<?, ? examples/s]

Map (num_proc=5):   0%|          | 0/4495 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,495 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 14,901,248 of 4,314,980,720 (0.35% trained)


Step,Training Loss
1,0.923100
2,1.111900
3,1.073500
4,1.181100
5,1.025000
6,0.941200
7,1.043600
8,0.939100
9,0.989000
10,0.882200


## 6. Test rápido

Antes de subir el modelo a HuggingFace, hago una prueba rápida para ver si al menos genera algo coherente. No es una evaluación formal — eso lo hago en el notebook 3.

In [16]:
messages = [{"role": "user", "content": [{"type": "text", "text": "¿Cómo puedo abrir una cuenta de ahorro?"}]}]
text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text=text, return_tensors="pt").to("cuda"),
    max_new_tokens=128,
    temperature=1.0,
    top_p=0.95,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)

Estoy aquí para ayudarte a abrir una cuenta de ahorro. Abrir una cuenta de ahorro puede ser un proceso simple. Esto es lo que necesitas hacer: 1. Visita la página web de nuestro banco de confianza o baja de tu aplicación móvil. 2. Busca la sección "Servicios" o "Cuenta de Banco" y ponle el nombre del servicio que te intere sa, tales como "Cuenta de Ahorro". 3. Haz clic en el nombre del servicio para comenzar el proceso de apertura de cuenta. 4. Te guiarán a través del proceso, que puede incluir un formulario que requiere información como


In [17]:
# Guardar adapter LoRA localmente
model.save_pretrained("banking-assistant-gemma3")
tokenizer.save_pretrained("banking-assistant-gemma3")

# Push a HuggingFace Hub (token desde Colab Secrets → clave "HF_TOKEN")
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

HF_USERNAME = "testlegadoss"
model.push_to_hub(f"{HF_USERNAME}/gemma3-4b-banking-assistant-es", token=hf_token)
tokenizer.push_to_hub(f"{HF_USERNAME}/gemma3-4b-banking-assistant-es", token=hf_token)

print(f"Modelo subido a: https://huggingface.co/{HF_USERNAME}/gemma3-4b-banking-assistant-es")

README.md:   0%|          | 0.00/572 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 17.0kB / 59.7MB            

Saved model to https://huggingface.co/testlegadoss/gemma3-4b-banking-assistant-es


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...panfipbnd/tokenizer.model: 100%|##########| 4.69MB / 4.69MB            

  ...mpanfipbnd/tokenizer.json: 100%|##########| 33.4MB / 33.4MB            

Modelo subido a: https://huggingface.co/testlegadoss/gemma3-4b-banking-assistant-es


## Conclusiones — Fase Modelo + Adaptación

- Modelo base: Gemma 3 4B Instruct (quantizado a 4-bit)
- Método: QLoRA con Unsloth (r=8, alpha=8)
- Parámetros entrenables: ~1-2% del total
